# 🐍 Python List Comprehensions — The Master Guide
*From Zero to Interview-Ready*

---

## Mental Model — The Conveyor Belt

A list comprehension is a factory conveyor belt written in one line. Raw materials (the iterable) go in on the right, a filter gate decides what passes through, a transformer stamps each piece, and finished products drop into a new list on the left. You read it right to left: 'for each x in data — if it passes the gate — apply the stamp.' The for-loop does the same job in four lines; the comprehension does it in one.

---

## Table of Contents

| # | Section | Link |
|---|---------|------|
| 1 | What Is a List Comprehension? The Visual Model | [→ #1](#1) |
| 2 | Creating / Setup | [→ #2](#2) |
| 3 | The Core API — Syntax Patterns | [→ #3](#3) |
| 4 | Decision Map — When To Use What | [→ #4](#4) |
| 5 | Pattern 1: Basic Transform — for-loop vs comprehension side-by-side (LC 1, 347) | [→ #5](#5) |
| 6 | Pattern 2: Filter + Transform — [expr for x in data if cond] (LC 15, 56) | [→ #6](#6) |
| 7 | Pattern 3: Nested Comprehension — 2D matrix flatten and transform (LC 48, 54) | [→ #7](#7) |
| 8 | Pattern 4: Comprehension over enumerate/zip — index-aware transforms (LC 238, 49) | [→ #8](#8) |
| 9 | Pattern 5: Dict and Set Comprehensions — {k:v} and {x} idioms (LC 49, 1) | [→ #9](#9) |
| 10 | The List Comprehensions Decision Map | [→ #10](#10) |
| 11 | Interview Cheat Sheet | [→ #11](#11) |

<a id='1'></a>

## 1. What Is a List Comprehension? The Visual Model

```
              LIST COMPREHENSION — THE CONVEYOR BELT

  RAW MATERIALS                                         FINISHED PRODUCTS
  (iterable)                                            (new list)

  [1, 2, 3, 4, 5, 6, 7, 8]
       │
       ▼
  ┌─────────────┐
  │  for x in   │  ← picks up each item one at a time
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │  if x % 2   │  ← filter gate — only even numbers pass
  │   == 0      │
  └──────┬──────┘
         │  2  4  6  8  (odd numbers were dropped)
         ▼
  ┌─────────────┐
  │   x * x     │  ← transformer — square each piece
  └──────┬──────┘
         │
         ▼
  [4, 16, 36, 64]   ← new list — original untouched

  PYTHON: result = [x*x for x in data if x % 2 == 0]

  READ RIGHT TO LEFT:
  "for each x in data — if x is even — compute x*x — collect into list"

  EQUIVALENT FOR-LOOP (4 lines → 1 line):
  result = []
  for x in data:
      if x % 2 == 0:
          result.append(x * x)

  FOR-LOOP vs COMPREHENSION PERFORMANCE
  ──────────────────────────────────────
  Comprehension runs in C under the hood — roughly 10–30% faster than
  the equivalent for-loop for simple transforms.
  Use timeit to verify: %timeit [x*x for x in range(10000)]
  Do NOT use comprehensions for side effects (printing, mutating state).
  A comprehension is for producing a new collection — nothing else.
```

<a id='2'></a>

## 2. Creating / Setup

In [ ]:
# ── EVERY FORM OF LIST COMPREHENSION ─────────────────────

data = [1, 2, 3, 4, 5, 6, 7, 8]

# basic: transform every element
squares    = [x * x         for x in data]                     # square each element
doubled    = [x * 2         for x in data]                     # double each element
as_strings = [str(x)        for x in data]                     # convert to string

# with filter: transform only matching elements
evens      = [x             for x in data if x % 2 == 0]       # keep evens only
even_sq    = [x * x         for x in data if x % 2 == 0]       # square evens only
big        = [x             for x in data if x > 5]            # keep > 5 only

# from a string
chars      = [c.upper()     for c in "hello"]                  # ['H','E','L','L','O']
no_vowels  = [c             for c in "hello" if c not in "aeiou"]  # consonants only

# from range
first_10   = [x             for x in range(10)]                # 0..9
odd_range  = [x             for x in range(20) if x % 2 != 0] # odd numbers 1..19

print(f"squares    : {squares}")
print(f"doubled    : {doubled}")
print(f"evens      : {evens}")
print(f"even_sq    : {even_sq}")
print(f"big        : {big}")
print(f"chars      : {chars}")
print(f"no_vowels  : {no_vowels}")
print(f"first_10   : {first_10}")
print(f"odd_range  : {odd_range}")

# side-by-side: for-loop and comprehension doing the same job
result_loop = []
for x in data:
    if x % 2 == 0:
        result_loop.append(x * x)

result_comp = [x * x for x in data if x % 2 == 0]

print(f"loop result : {result_loop}")
print(f"comp result : {result_comp}")
print(f"same output : {result_loop == result_comp}")   # True

# Simplicity and clarity is Gold

<a id='3'></a>

## 3. The Core API — Syntax Patterns

```
SYNTAX PATTERN                                WHAT IT DOES
────────────────────────────────────────────────────────────────────────────────
[expr for x in iterable]                      basic transform — every element
[expr for x in iterable if cond]              filter + transform
[expr if cond else other for x in iterable]   ternary — transform ALL, two branches
[f(x) for x in iterable]                      call function on each element
[expr for x in a for y in b]                  nested loop — outer first, inner second
[expr for row in matrix for x in row]         flatten 2D to 1D
[row[:] for row in matrix]                    copy each row — safe 2D copy
{k: v for k, v in pairs}                      dict comprehension
{x for x in iterable}                         set comprehension — deduplicates
(x for x in iterable)                         generator expression — lazy, no list

READING ORDER
─────────────────────────────────────────────────────────────────
result = [  transform(x)  for x in data  if keep(x)  ]
            ─────────────  ────────────   ──────────
            WHAT TO DO     WHERE TO GET   FILTER
            (leftmost)     (middle)       (rightmost, optional)

Read: "for each x in data, if keep(x), do transform(x), collect."

TERNARY FORM (no filter — replaces every element)
─────────────────────────────────────────────────
[x if x > 0 else 0 for x in nums]   ← relu: negative → 0, positive → keep
["even" if x%2==0 else "odd" for x in nums]

NESTED COMPREHENSION READING ORDER
───────────────────────────────────────────────────────────────────────────────
[cell for row in matrix for cell in row]
  outer loop first (for row in matrix)
  inner loop second (for cell in row)
  same order as the equivalent nested for-loops

THINGS YOU DO NOT DO
────────────────────────────────────────────────────────────
❌  [print(x) for x in data]         side effect in comprehension — use a for-loop
❌  [a.append(x) for x in data]      mutating inside comprehension — silent bug
❌  [x for x in data if complex_logic_with_5_conditions]  — unreadable, use a loop
❌  nested comprehension 3+ levels deep  — cognitive overload, extract a function
❌  [f(x) for x in data] when f() is expensive and result unused  — use loop + explicit logging
✅  [x for x in data if cond]        filter
✅  [f(x) for x in data]             transform
✅  {k:v for k,v in d.items()}       rebuild dict
✅  [row[:] for row in matrix]       copy 2D list rows safely
```

In [ ]:
# ── CORE API DEMO — run this cell, read every print ──────

nums = [1, -2, 3, -4, 5, -6]

# ── BASIC TRANSFORM ──────────────────────────────────────
print([x * 2        for x in nums])              # doubled
print([abs(x)       for x in nums])              # absolute value

# ── FILTER ───────────────────────────────────────────────
print([x            for x in nums if x > 0])     # positives only
print([x * x        for x in nums if x < 0])     # square the negatives

# ── TERNARY — no filter, two branches for every element ──
print([x if x >= 0 else 0 for x in nums])        # relu: clamp negatives to 0
print(["+" if x > 0 else "-" for x in nums])     # sign label

# ── FUNCTION CALL ─────────────────────────────────────────
print([str(x)       for x in nums])              # convert all to string
print([abs(x) * 2   for x in nums])              # chain operations

# ── NESTED — iterate two iterables together ───────────────
pairs = [(x, y) for x in [1, 2] for y in [10, 20]]
print(f"pairs: {pairs}")                          # [(1,10),(1,20),(2,10),(2,20)]

# ── FLATTEN 2D → 1D ──────────────────────────────────────
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat   = [cell for row in matrix for cell in row]
print(f"flat: {flat}")                            # [1,2,3,4,5,6,7,8,9]

# ── SAFE 2D COPY ─────────────────────────────────────────
copy = [row[:] for row in matrix]                # each row is a new list
copy[0][0] = 99
print(f"matrix[0][0] after copy mutated: {matrix[0][0]}")  # 1 — original safe

# ── DICT COMPREHENSION ────────────────────────────────────
word_len = {w: len(w) for w in ["apple", "fig", "banana"]}
print(f"word lengths: {word_len}")

# ── SET COMPREHENSION ─────────────────────────────────────
unique_lens = {len(w) for w in ["apple", "fig", "banana", "ant"]}
print(f"unique lengths: {unique_lens}")

# Simplicity and clarity is Gold

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                        WHAT TO DO
──────────────────────────────────────────────────────────────────────────────
transform every element of a list            [f(x) for x in data]
keep only elements matching a condition      [x for x in data if cond(x)]
transform + filter in one pass               [f(x) for x in data if cond(x)]
need both branches (transform all)           [a if cond else b for x in data]
flatten a 2D matrix to 1D                    [x for row in matrix for x in row]
copy each row of a 2D list safely            [row[:] for row in matrix]
build a frequency map / grouping dict        {k: f(v) for k, v in pairs}
deduplicate + transform                      {f(x) for x in data}
transform using index                        [f(i,x) for i, x in enumerate(data)]
pair two lists together                      [f(a,b) for a, b in zip(list1, list2)]
collect keys from a list of dicts            [d["key"] for d in records]
top-k collect after sort                     [x for x in sorted(data)[-k:]]
need lazy evaluation (large input)           use (x for x in data) generator instead
side effects needed                          use a for-loop — NOT a comprehension
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Basic Transform — for-loop vs comprehension side-by-side — LC 1, 347

```
PROBLEM:
  LC 347 — Top K Frequent Elements
  Given an integer array nums and integer k, return the k most frequent elements.

COMPREHENSION ROLE:
  After building a frequency dict, collect the top-k keys in one line:
    result = [item for item, _ in counter.most_common(k)]
  Without comprehension:
    result = []
    for item, _ in counter.most_common(k):
        result.append(item)

  The comprehension is not just shorter — it expresses INTENT:
  "collect the item (not the count) from each (item, count) pair."

SLOW MOTION TRACE on nums = [1,1,1,2,2,3], k=2:
  counter = {1:3, 2:2, 3:1}
  counter.most_common(2) = [(1,3), (2,2)]

  for-loop trace:
  step 1: item=1, _=3  → append 1  result=[1]
  step 2: item=2, _=2  → append 2  result=[1,2]

  comprehension: [item for item, _ in [(1,3),(2,2)]]
  reads: "for each (item,count) pair, take item, collect"
  result: [1, 2]  ✓

KEY INSIGHT:
  _ is convention for "I don't need this value."
  The comprehension reads like the English description of what you want.

TIME / SPACE:
  Time:  O(n log n) — dominated by most_common (heap internally)
  Space: O(n)       — frequency dict + result list
```

In [ ]:
from collections import Counter

def top_k_frequent(nums: list, k: int) -> list:
    """
    LC 347 — Top K Frequent Elements
    Approach: Counter.most_common(k) then collect keys via comprehension.
    Args:
        nums (list[int]): integer array.
        k (int): number of most frequent elements to return.
    Returns:
        list[int]: k most frequent elements, any order.
    Time:  O(n log n) — Counter build O(n), most_common O(n log k)
    Space: O(n)       — counter stores at most n distinct elements
    """
    counter = Counter(nums)        # build frequency map in one shot

    # comprehension version — express intent: "take the item, not the count"
    # slow motion on nums=[1,1,1,2,2,3], k=2:
    # counter.most_common(2) = [(1,3), (2,2)]
    # step 1: item=1, _=3  → collect 1
    # step 2: item=2, _=2  → collect 2
    # result: [1, 2]
    result = [item for item, _ in counter.most_common(k)]

    return result


def top_k_loop_version(nums: list, k: int) -> list:
    """Same as top_k_frequent but using explicit for-loop — for comparison."""
    counter = Counter(nums)
    result = []                    # 4 lines to say what the comprehension says in 1
    for item, _ in counter.most_common(k):
        result.append(item)
    return result


def side_by_side_demo():
    """Show for-loop and comprehension producing identical output."""
    data  = [1, 1, 2, 2, 2, 3, 3, 3, 3]
    k     = 2

    loop_result = top_k_loop_version(data, k)
    comp_result = top_k_frequent(data, k)

    print(f"data         : {data}")
    print(f"loop result  : {loop_result}")
    print(f"comp result  : {comp_result}")
    print(f"identical    : {set(loop_result) == set(comp_result)}")


def test_harness(fn):
    tests = [
        ([1,1,1,2,2,3], 2, {1, 2}),
        ([1],           1, {1}),
        ([1,2],         2, {1, 2}),
        ([4,4,4,3,3,2], 2, {4, 3}),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = set(fn(*inputs))           # order doesn't matter
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(top_k_frequent)
side_by_side_demo()

print("top_k_frequent defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Filter + Transform — [expr for x in data if cond] — LC 15, 56

```
PROBLEM:
  LC 56 — Merge Intervals
  Given a list of intervals, merge all overlapping intervals.

COMPREHENSION ROLE:
  After sorting, collect only intervals that do NOT overlap with the current merged one:
  or more commonly — collect starts and ends separately for analysis:
    starts = [iv[0] for iv in intervals]
    ends   = [iv[1] for iv in intervals]

  Filter + transform in the output phase:
    result = [[s, e] for s, e in merged_pairs if s <= e]

FILTER vs TERNARY — know which tool to pick:
  FILTER — [x for x in data if cond]        → result may be shorter than input
  TERNARY — [a if cond else b for x in data] → result is ALWAYS same length as input

  LC 56 merging uses filter: some intervals get absorbed — output shorter than input.
  Relu-style clamping uses ternary: every element transforms — output same length.

SLOW MOTION TRACE on intervals = [[1,3],[2,6],[8,10],[15,18]]:
  sorted: [[1,3],[2,6],[8,10],[15,18]]
  merged: start with [1,3]
    [2,6]:  2<=3 → overlap → extend to [1,6]
    [8,10]: 8>6  → no overlap → emit [1,6], start new [8,10]
    [15,18]:15>10→ no overlap → emit [8,10], start new [15,18]
  emit final [15,18]
  result: [[1,6],[8,10],[15,18]]

  comprehension collects starts to check sorting:
    starts = [iv[0] for iv in intervals]  → [1, 2, 8, 15]

KEY INSIGHT:
  Filter gate is evaluated BEFORE the transform — failed elements are never processed.
  This is why [f(x) for x in data if cond] is safe even when f(x) would crash on bad inputs.

TIME / SPACE:
  Time:  O(n log n) — sort dominates; merging and collecting are O(n)
  Space: O(n)       — output list
```

In [ ]:
def merge_intervals(intervals: list) -> list:
    """
    LC 56 — Merge Intervals
    Approach: sort by start, sweep and extend current window, collect non-overlapping.
    Args:
        intervals (list[list[int]]): list of [start, end] pairs.
    Returns:
        list[list[int]]: merged non-overlapping intervals.
    Time:  O(n log n) — sort + one linear sweep
    Space: O(n)       — output list
    """
    if not intervals:
        return []

    # sort by start time — comprehension extracts starts for a quick sanity check
    intervals.sort(key=lambda iv: iv[0])
    starts = [iv[0] for iv in intervals]     # [1, 2, 8, 15] — for debugging/tracing
    ends   = [iv[1] for iv in intervals]     # [3, 6, 10, 18]

    merged = [intervals[0][:]]               # start with first interval (copy it)

    for iv in intervals[1:]:
        # slow motion on [[1,3],[2,6],[8,10],[15,18]]:
        # iv=[2,6]:  2<=3 → overlap → merged[-1]=[1,6]
        # iv=[8,10]: 8>6  → new window → merged=[[1,6],[8,10]]
        # iv=[15,18]:15>10→ new window → merged=[[1,6],[8,10],[15,18]]
        if iv[0] <= merged[-1][1]:           # overlap: new start ≤ current end
            merged[-1][1] = max(merged[-1][1], iv[1])   # extend end
        else:
            merged.append(iv[:])             # no overlap — start new window

    return merged


def filter_transform_demo():
    """Show filter vs ternary — two different tools, two different behaviors."""
    nums = [-3, -1, 0, 2, 4, -5, 7]

    # FILTER — output shorter than input
    positives = [x for x in nums if x > 0]
    print(f"filter  (positives only) : {positives}   len={len(positives)}")

    # TERNARY — output same length as input
    clamped = [x if x >= 0 else 0 for x in nums]
    print(f"ternary (clamp to 0)     : {clamped}  len={len(clamped)}")

    print(f"original len={len(nums)}, filter len={len(positives)}, ternary len={len(clamped)}")


def test_harness(fn):
    tests = [
        ([[1,3],[2,6],[8,10],[15,18]],  [[1,6],[8,10],[15,18]]),
        ([[1,4],[4,5]],                 [[1,5]]),
        ([[1,4],[2,3]],                 [[1,4]]),
        ([[1,2]],                       [[1,2]]),
        ([[1,4],[0,2],[3,5]],           [[0,5]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        import copy
        got = fn(copy.deepcopy(inputs[0]))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(merge_intervals)
filter_transform_demo()

print("merge_intervals defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Nested Comprehension — 2D matrix flatten and transform — LC 48, 54

```
PROBLEM:
  LC 54 — Spiral Matrix: collect all elements — output is 1D
  LC 48 — Rotate Image: transpose rows ↔ cols then reverse each row

NESTED COMPREHENSION SYNTAX:
  [expr  for outer in outer_iter  for inner in inner_iter]
   ^      ^─────────────────────  ^───────────────────────
   stamp  outer loop (goes first) inner loop (goes second)

  Same order as the equivalent for-loops:
    for outer in outer_iter:        ← outer loop always goes FIRST
        for inner in inner_iter:    ← inner loop always goes SECOND
            result.append(expr)

2D FLATTEN:
  flat = [cell for row in matrix for cell in row]
  Reads: "for each row in matrix, for each cell in row, collect cell."
  [[1,2],[3,4]] → [1, 2, 3, 4]

MATRIX COMPREHENSION (build a new 2D grid):
  transposed = [[matrix[r][c] for r in range(rows)]
                               for c in range(cols)]
  Outer comp (for c): builds one NEW ROW per column
  Inner comp (for r): fills that row with elements from original rows

SAFE 2D COPY:
  copy = [row[:] for row in matrix]   ← each row is an independent new list
  copy[0][0] = 99                     ← does NOT change original matrix

SLOW MOTION TRACE — flatten [[1,2,3],[4,5,6]]:
  outer=row=[1,2,3]  inner=cell=1 → collect 1
                     inner=cell=2 → collect 2
                     inner=cell=3 → collect 3
  outer=row=[4,5,6]  inner=cell=4 → collect 4
                     inner=cell=5 → collect 5
                     inner=cell=6 → collect 6
  result: [1, 2, 3, 4, 5, 6]

KEY INSIGHT:
  The for-clauses in a nested comprehension run LEFT TO RIGHT, same order as
  the equivalent for-loops written top to bottom. Never reverse this mentally.

TIME / SPACE:
  Time:  O(m*n) — every element touched once
  Space: O(m*n) — new output list
```

In [ ]:
def rotate_with_comp(matrix: list) -> None:
    """
    LC 48 — Rotate Image (90° clockwise) using nested comprehension for transpose.
    Approach: build transposed matrix with nested comprehension, then reverse each row.
    Args:
        matrix (list[list[int]]): n×n matrix, modified in-place.
    Returns:
        None: mutates matrix directly.
    Time:  O(n²) — transpose touches every element
    Space: O(n²) — temporary transposed matrix
    """
    n = len(matrix)

    # ── STEP 1: TRANSPOSE using nested comprehension ──
    # transposed[c][r] = matrix[r][c]
    # outer comp: for each column c, build one new row
    # inner comp: for each row r, grab matrix[r][c]
    # slow motion on n=3:
    # c=0: [matrix[0][0], matrix[1][0], matrix[2][0]] = [1, 4, 7]
    # c=1: [matrix[0][1], matrix[1][1], matrix[2][1]] = [2, 5, 8]
    # c=2: [matrix[0][2], matrix[1][2], matrix[2][2]] = [3, 6, 9]
    transposed = [[matrix[r][c] for r in range(n)] for c in range(n)]

    # ── STEP 2: COPY BACK and REVERSE EACH ROW ──
    for r in range(n):
        matrix[r] = transposed[r][::-1]    # write reversed transposed row back


def nested_comp_demo():
    """Show flatten, transform, safe copy — all nested comprehension patterns."""
    matrix = [[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]]

    # flatten 2D → 1D
    flat = [cell for row in matrix for cell in row]
    print(f"flatten       : {flat}")

    # flatten + filter: only even elements
    even_flat = [cell for row in matrix for cell in row if cell % 2 == 0]
    print(f"even flatten  : {even_flat}")

    # transform every element: square each cell
    squared = [[cell * cell for cell in row] for row in matrix]
    print(f"squared 2D    : {squared}")

    # safe 2D copy
    safe_copy = [row[:] for row in matrix]
    safe_copy[0][0] = 99
    print(f"matrix[0][0] after copy mutated: {matrix[0][0]}")   # 1 — unchanged


def test_harness(fn):
    import copy
    tests = [
        ([[1,2,3],[4,5,6],[7,8,9]],                       [[7,4,1],[8,5,2],[9,6,3]]),
        ([[5,1,9,11],[2,4,8,10],[13,3,6,7],[15,14,12,16]],[[15,13,2,5],[14,3,4,1],[12,6,8,9],[16,7,10,11]]),
        ([[1]],                                            [[1]]),
        ([[1,2],[3,4]],                                    [[3,1],[4,2]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        matrix = copy.deepcopy(inputs[0])
        fn(matrix)
        got = matrix
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(rotate_with_comp)
nested_comp_demo()

print("rotate_with_comp defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Comprehension over enumerate/zip — index-aware transforms — LC 238, 49

```
PROBLEM:
  LC 238 — Product of Array Except Self: build prefix and suffix lists
  LC 49  — Group Anagrams: group words by their sorted-letter signature

ENUMERATE — when you need both index and value:
  [(i, x) for i, x in enumerate(data)]
  [x * i  for i, x in enumerate(data)]      ← weight by position
  [i      for i, x in enumerate(data) if x > 0]  ← indices of positives

ZIP — when you need to pair two lists element-by-element:
  [a + b  for a, b in zip(prefix, suffix)]  ← LC 238 product
  [a * b  for a, b in zip(list1, list2)]    ← elementwise multiply
  [(a, b) for a, b in zip(keys, values)]    ← pair up

COMBINING WITH FILTER:
  [x for i, x in enumerate(data) if i % 2 == 0]   ← even-indexed elements
  [x for i, x in enumerate(data) if i < len(data)//2]  ← first half

SLOW MOTION TRACE — prefix * suffix product for nums=[1,2,3,4]:
  prefix = [1, 1, 2,  6]    (product of everything to the left)
  suffix = [24,12, 4,  1]    (product of everything to the right)
  result = [a*b for a,b in zip(prefix,suffix)]
  step 1: a=1,  b=24 → 24
  step 2: a=1,  b=12 → 12
  step 3: a=2,  b=4  → 8
  step 4: a=6,  b=1  → 6
  result: [24, 12, 8, 6]  ✓

KEY INSIGHT:
  zip() is the safe way to pair two lists — it stops at the shorter one.
  Never use range(len(a)) when zip(a, b) expresses the intent more clearly.

TIME / SPACE:
  Time:  O(n) — one pass through the zipped pairs
  Space: O(n) — output list
```

In [ ]:
from collections import defaultdict

def group_anagrams(strs: list) -> list:
    """
    LC 49 — Group Anagrams
    Approach: sort each word's letters to get a signature, group by signature.
    Args:
        strs (list[str]): list of lowercase words.
    Returns:
        list[list[str]]: groups of anagrams, any order within groups.
    Time:  O(n * k log k) — n words, each sorted in O(k log k)
    Space: O(n * k)       — store all words in the grouping dict
    """
    groups = defaultdict(list)

    for word in strs:
        # signature = sorted letters joined — "eat","tea","ate" all → "aet"
        # comprehension inside join: [c for c in sorted(word)]
        # or more directly: "".join(sorted(word))
        sig = "".join(sorted(word))         # sort letters → canonical form
        groups[sig].append(word)            # group by signature

    # dict comprehension to see the grouping (for display)
    sig_map = {sig: words for sig, words in groups.items()}

    # collect groups — comprehension extracts the values
    return [words for words in groups.values()]


def enumerate_zip_demo():
    """Show enumerate and zip patterns with comprehensions."""
    data = [10, 20, 30, 40, 50]

    # enumerate: index + value
    indexed = [(i, x) for i, x in enumerate(data)]
    print(f"indexed pairs    : {indexed}")

    # enumerate + filter: indices where value > 25
    big_indices = [i for i, x in enumerate(data) if x > 25]
    print(f"indices where >25: {big_indices}")

    # zip: pair two lists
    weights = [1, 2, 3, 4, 5]
    weighted = [x * w for x, w in zip(data, weights)]
    print(f"weighted product : {weighted}")

    # zip for prefix * suffix (LC 238 pattern)
    nums   = [1, 2, 3, 4]
    prefix = [1, 1, 2, 6]          # precomputed
    suffix = [24, 12, 4, 1]        # precomputed
    result = [p * s for p, s in zip(prefix, suffix)]
    print(f"prefix*suffix    : {result}")   # [24, 12, 8, 6]

    # even-indexed elements only
    even_idx = [x for i, x in enumerate(data) if i % 2 == 0]
    print(f"even-indexed     : {even_idx}")


def test_harness(fn):
    def normalize(groups):
        return sorted([sorted(g) for g in groups])

    tests = [
        (["eat","tea","tan","ate","nat","bat"], [["ate","eat","tea"],["bat"],["nat","tan"]]),
        ([""],                                  [[""]]),
        (["a"],                                 [["a"]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got      = normalize(fn(*inputs))
        exp_norm = normalize(expected)
        status = "PASSED" if got == exp_norm else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={exp_norm} | got={got}")
        passed += (got == exp_norm)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(group_anagrams)
enumerate_zip_demo()

print("group_anagrams defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Dict and Set Comprehensions — {k:v} and {x} idioms — LC 49, 1

```
PROBLEM:
  LC 1   — Two Sum: build an index map {val → index} in one pass (or via comprehension)
  LC 49  — Group Anagrams: {signature → [word, word, ...]}

DICT COMPREHENSION — when you have pairs:
  {k: v    for k, v in pairs}           ← rebuild a dict from pairs
  {k: f(v) for k, v in d.items()}       ← transform values, keep keys
  {v: k    for k, v in d.items()}       ← invert a dict (assumes unique values)
  {x: x*x  for x in range(10)}          ← build from scratch
  {word: len(word) for word in words}   ← word → length map

SET COMPREHENSION — when you want unique values:
  {f(x)   for x in data}               ← unique transforms
  {x % 10 for x in nums}               ← unique last digits
  {w[0]   for w in words}              ← unique first letters

DICT vs LIST COMPREHENSION — choose based on OUTPUT SHAPE:
  need a list     → [...]    — ordered, allows duplicates
  need unique     → {...}    — unordered, no duplicates
  need key→value  → {k:v}   — lookup, O(1) access

SLOW MOTION TRACE — invert dict {1:"a", 2:"b", 3:"c"}:
  {v: k for k, v in {1:"a",2:"b",3:"c"}.items()}
  step 1: k=1, v="a" → {"a": 1}
  step 2: k=2, v="b" → {"a":1, "b":2}
  step 3: k=3, v="c" → {"a":1, "b":2, "c":3}
  result: {"a":1, "b":2, "c":3}  ✓

TWO SUM INDEX MAP (LC 1):
  index_map = {val: i for i, val in enumerate(nums)}
  Then for each num: check (target - num) in index_map.
  Comprehension builds the lookup table in one readable line.

KEY INSIGHT:
  Dict comprehension answers "what is the value for key X?"
  Set comprehension answers "have I seen X before?"
  They are the O(1) lookup builders — replace O(n) linear scans.

TIME / SPACE:
  Time:  O(n) — one pass to build
  Space: O(n) — dict or set stores at most n entries
```

In [ ]:
def two_sum(nums: list, target: int) -> list:
    """
    LC 1 — Two Sum
    Approach: one-pass hash map — build index map as we scan, check complement.
    Args:
        nums (list[int]): integer array.
        target (int): target sum.
    Returns:
        list[int]: [i, j] indices where nums[i] + nums[j] == target.
    Time:  O(n) — single pass
    Space: O(n) — hash map stores at most n entries
    """
    seen = {}                       # val → index map, built incrementally

    for i, num in enumerate(nums):
        complement = target - num   # what do we need to find?

        # slow motion on nums=[2,7,11,15], target=9:
        # i=0 num=2  complement=7  seen={}       7 not found  seen={2:0}
        # i=1 num=7  complement=2  seen={2:0}    2 found at 0 → return [0,1]
        if complement in seen:
            return [seen[complement], i]        # found the pair

        seen[num] = i               # record this value's index for future complements

    return []


def dict_set_comp_demo():
    """Demonstrate all dict and set comprehension patterns."""
    words = ["apple", "fig", "banana", "ant", "fig", "cherry"]

    # dict: word → length
    word_len = {w: len(w) for w in words}
    print(f"word→len       : {word_len}")

    # dict: invert — length → list of words at that length (non-trivial invert)
    # simple invert (assumes unique values):
    d = {1: "a", 2: "b", 3: "c"}
    inverted = {v: k for k, v in d.items()}
    print(f"inverted dict  : {inverted}")

    # dict: transform values
    upper_len = {w: len(w) * 2 for w, _ in word_len.items()}
    print(f"double lengths : {upper_len}")

    # dict: filter keys
    long_words = {w: l for w, l in word_len.items() if l > 3}
    print(f"long words     : {long_words}")

    # set: unique lengths
    unique_lens = {len(w) for w in words}
    print(f"unique lengths : {unique_lens}")

    # set: unique first letters
    first_letters = {w[0] for w in words}
    print(f"first letters  : {first_letters}")

    # index map (LC 1 pattern)
    nums = [4, 7, 2, 9, 1]
    index_map = {val: i for i, val in enumerate(nums)}
    print(f"index map      : {index_map}")


def test_harness(fn):
    tests = [
        ([2, 7, 11, 15], 9,  [0, 1]),
        ([3, 2, 4],      6,  [1, 2]),
        ([3, 3],         6,  [0, 1]),
        ([1, 5, 3, 7],   8,  [1, 3]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(two_sum)
dict_set_comp_demo()

print("two_sum defined.")

<a id='10'></a>

## 10. The List Comprehensions Decision Map

```
QUESTION TYPE                              KEY TECHNIQUE                    LC PROBLEMS
────────────────────────────────────────────────────────────────────────────────────────
collect top-k items from a sorted sequence [item for item,_ in most_common(k)]  347
flatten 2D list to 1D                      [x for row in m for x in row]    54, 48
filter elements matching a condition       [x for x in data if cond(x)]     56, 435
transform without filtering                [f(x) for x in data]             238, 1
filter + transform in one pass             [f(x) for x in data if cond(x)]  15, 56
both branches (replace every element)      [a if c else b for x in data]    —
pair two lists elementwise                 [f(a,b) for a,b in zip(l1,l2)]   238, 560
index + value together                     [f(i,x) for i,x in enumerate(d)] 238, 49
safe 2D copy (independent rows)            [row[:] for row in matrix]       48, 54
transpose a 2D matrix                      [[m[r][c] for r in R] for c in C]  48
group by key into dict                     {k:v for k,v in pairs}           49, 1
deduplicate + transform                    {f(x) for x in data}             —
build index map for O(1) lookup            {val:i for i,val in enumerate(a)} 1
```

<a id='11'></a>

## 11. Interview Cheat Sheet

### 1. When to reach for a list comprehension

| Signal | What to Do |
|--------|------------|
| "transform every element" | `[f(x) for x in data]` |
| "keep only elements where..." | `[x for x in data if cond(x)]` |
| "collect the top-k..." | `[item for item, _ in counter.most_common(k)]` |
| "flatten the matrix" | `[x for row in matrix for x in row]` |
| "pair two arrays" | `[f(a,b) for a, b in zip(list1, list2)]` |
| "need index and value" | `[f(i,x) for i, x in enumerate(data)]` |
| "build a lookup table" | `{val: i for i, val in enumerate(nums)}` |
| "group by some key" | `{k: v for k, v in pairs}` |
| "deduplicate the results" | `{f(x) for x in data}` |

### 2. The core patterns — memorize these

```python
# BASIC TRANSFORM
result = [f(x) for x in data]

# FILTER ONLY
result = [x for x in data if cond(x)]

# FILTER + TRANSFORM
result = [f(x) for x in data if cond(x)]

# TERNARY (both branches, same length output)
result = [a if cond(x) else b for x in data]

# NESTED (flatten or build 2D)
flat = [x for row in matrix for x in row]
grid = [[f(r,c) for c in range(cols)] for r in range(rows)]

# DICT / SET COMPREHENSION
d = {k: v for k, v in pairs}
s = {f(x) for x in data}
```

### 3. Common templates

```python
# TOP-K COLLECT — LC 347
result = [item for item, _ in Counter(nums).most_common(k)]

# FLATTEN 2D — LC 54
flat = [cell for row in matrix for cell in row]

# TRANSPOSE 2D — LC 48
transposed = [[matrix[r][c] for r in range(n)] for c in range(n)]

# PREFIX * SUFFIX VIA ZIP — LC 238
result = [p * s for p, s in zip(prefix, suffix)]

# INDEX MAP — LC 1
index_map = {val: i for i, val in enumerate(nums)}
```

### 4. Gotchas

```
❌  [print(x) for x in data]          side effect in comp — use for-loop
❌  [a.append(x) for x in data]       mutating inside comp — always wrong
❌  nested comp 3+ levels deep        unreadable — extract helper function
❌  [f(x) for x in data if cond][0]   crashes if empty — check first
❌  {v:k for k,v in d.items()} if values not unique  last one wins silently
✅  [f(x) for x in data if cond(x)]  filter gate runs BEFORE transform
✅  for outer first, inner second — same order as nested for-loops
✅  _ convention for unused variable: [item for item, _ in pairs]
✅  [row[:] for row in matrix] — safe 2D row copy, not aliased rows
✅  (x for x in data) — generator, lazy — use when input is huge
```

## 12. Summary

```
              🐍 LIST COMPREHENSIONS — MASTER MAP

              [expr  for x in iterable  if cond]
                        │
          ┌─────────────┼─────────────────────────────┐
          │             │                             │
      BASIC         FILTER                       NESTED
    [f(x)           [x if cond]              [x for row in m
     for x in]      [f(x) if cond]            for x in row]
          │             │                  [f(r,c) for c in C]
          │             │                   for r in R]
      LC 347         LC 56 merge               LC 48 rotate
      top-k          LC 435 intervals          LC 54 spiral
      LC 238         LC 15 triplets
      prefix/suf          │
          │           TERNARY
          │          [a if c else b
          │           for x in data]
      ENUMERATE / ZIP    │
    [f(i,x) for i,x    DICT / SET
      in enumerate]    {k:v for k,v}
    [f(a,b) for a,b    {x for x in}
      in zip(l1,l2)]      │
          │            LC 49 anagrams
      LC 238 prefix    LC 1 two sum
      LC 49 grouping   index maps
```

---
*End of List Comprehensions Master Guide — Sean Edition*